# 🔀 Multimodal: Zero-shot vs. Few-shot (CLIP)

**CO5085 – Assignment 1**

Dùng CLIP để so sánh zero-shot và few-shot classification trên tập đa phương thức (Flickr30k/COCO subset).

In [ ]:
import sys
sys.path.insert(0, '../src')
import torch, clip, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import os
os.makedirs('../results', exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model, preprocess = clip.load('ViT-B/32', device=DEVICE)
print(f'CLIP loaded on {DEVICE} ✅')

## 1. Dataset Preparation

Sử dụng CIFAR-100 super-classes (20 classes) làm **proxy multimodal dataset** với:
- Ảnh từ CIFAR-100
- Text prompt tự động: `'a photo of a {superclass}'`

> Nếu có Flickr30k, thay bằng dataset đó.

In [ ]:
from torchvision import datasets
import torchvision.transforms as T

# CIFAR-100 superclasses (20 groups)
SUPERCLASSES = [
    'aquatic mammals', 'fish', 'flowers', 'food containers', 'fruit and vegetables',
    'household electrical devices', 'household furniture', 'insects', 'large carnivores',
    'large man-made outdoor things', 'large natural outdoor scenes', 'large omnivores and herbivores',
    'medium-sized mammals', 'non-insect invertebrates', 'people', 'reptiles',
    'small mammals', 'trees', 'vehicles 1', 'vehicles 2'
]

# CIFAR-100 maps each fine class to a superclass
coarse_labels = [
    4, 1, 14, 8, 0, 6, 7, 7, 18, 3, 3, 14, 9, 18, 7, 11, 3, 9, 7, 11,
    6, 11, 5, 10, 7, 6, 13, 15, 3, 15, 0, 11, 1, 10, 12, 14, 16, 9, 11, 5,
    5, 19, 8, 8, 15, 13, 14, 17, 18, 10, 16, 4, 17, 4, 2, 0, 17, 4, 18, 17,
    10, 3, 2, 12, 12, 16, 12, 1, 9, 19, 2, 10, 0, 1, 16, 12, 9, 13, 15, 13,
    16, 19, 2, 4, 6, 19, 5, 5, 8, 19, 18, 1, 2, 15, 6, 0, 17, 8, 14, 13
]

test_ds = datasets.CIFAR100(root='../data/image', train=False, download=True,
                             transform=T.Compose([T.Resize(224), T.ToTensor()]))
test_images = [T.ToPILImage()(test_ds[i][0]) for i in range(500)]
test_labels = [coarse_labels[test_ds[i][1]] for i in range(500)]
print(f'Test samples: {len(test_images)} | Superclasses: {len(SUPERCLASSES)}')

## 2. Zero-shot Classification

In [ ]:
from models import CLIPZeroShotClassifier

zero_clf = CLIPZeroShotClassifier(class_names=SUPERCLASSES, device=DEVICE)
BATCH = 50
all_preds = []
for i in range(0, len(test_images), BATCH):
    batch_imgs = test_images[i:i+BATCH]
    _, preds = zero_clf.predict(batch_imgs)
    all_preds.extend(preds.tolist())

from sklearn.metrics import accuracy_score, f1_score
acc_zs = accuracy_score(test_labels, all_preds)
f1_zs  = f1_score(test_labels, all_preds, average='macro', zero_division=0)
print(f'Zero-shot → Accuracy: {acc_zs:.4f} | F1-Macro: {f1_zs:.4f}')

## 3. Few-shot Classification (CLIP + Linear Probe)

In [ ]:
from models import CLIPFewShotClassifier
import torch.nn as nn
from torch.optim import Adam

def run_few_shot(k_shots_per_class: int):
    """Train linear probe with k examples per class."""
    clf = CLIPFewShotClassifier(num_classes=len(SUPERCLASSES), device=DEVICE).to(DEVICE)

    # Build few-shot training set
    from collections import defaultdict
    class_examples = defaultdict(list)
    for idx in range(len(test_ds)):
        img, fine_label = test_ds[idx]
        coarse = coarse_labels[fine_label]
        if len(class_examples[coarse]) < k_shots_per_class:
            class_examples[coarse].append(T.ToPILImage()(img))
    
    # Extract CLIP features
    support_imgs, support_labels = [], []
    for c, imgs in class_examples.items():
        support_imgs.extend(imgs)
        support_labels.extend([c] * len(imgs))
    
    X_support = clf.encode_images(support_imgs)
    y_support = torch.tensor(support_labels, dtype=torch.long).to(DEVICE)

    # Train linear head
    optimizer = Adam(clf.classifier.parameters(), lr=1e-2)
    criterion = nn.CrossEntropyLoss()
    for _ in range(200):
        optimizer.zero_grad()
        logits = clf.classifier(X_support)
        loss = criterion(logits, y_support)
        loss.backward()
        optimizer.step()

    # Evaluate
    clf.eval()
    test_feats = clf.encode_images(test_images)
    with torch.no_grad():
        logits = clf.classifier(test_feats)
        preds = logits.argmax(-1).cpu().numpy()

    acc = accuracy_score(test_labels, preds)
    f1  = f1_score(test_labels, preds, average='macro', zero_division=0)
    print(f'  {k_shots_per_class}-shot → Acc: {acc:.4f}, F1: {f1:.4f}')
    return acc, f1

results_mm = {'Zero-shot': {'accuracy': acc_zs, 'f1_macro': f1_zs}}
for k in [1, 5, 10, 20]:
    acc, f1 = run_few_shot(k)
    results_mm[f'{k}-shot'] = {'accuracy': acc, 'f1_macro': f1}

## 4. Comparison Chart

In [ ]:
from evaluate import compare_models
compare_models(results_mm, metric='accuracy', save_path='../results/multimodal_comparison_acc.png')